In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

import hamiltonian_generator

from qiskit.primitives import StatevectorEstimator
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2
from qiskit_ibm_runtime.fake_provider import FakeGuadalupeV2

from itertools import product

import warnings
warnings.filterwarnings("ignore")

In [2]:
def save_reference_energy(n, h) :
    ansatz = efficient_su2(num_qubits=n, reps=2)
    hamiltonian = hamiltonian_generator.get_heisenberg_hamiltonian(n, h)
    estimator = StatevectorEstimator()
    params = np.load(f'params/params_{n}_{h}.npy')
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )
    np.save(f'data/ref_{n}_{h}', energy)
    return energy

In [3]:
num_sites = [12, 8, 5, 3]
couplings = [0, 0.5, 2, 5]

for n, h in product(num_sites, couplings) :
    energy = save_reference_energy(n, h)
    print(f"n={n}, h={h} \t\t {energy}")

n=12, h=0 		 [-17.23164431]
n=12, h=0.5 		 [-16.24323482]
n=12, h=2 		 [-20.07379294]
n=12, h=5 		 [-48.92667173]
n=8, h=0 		 [-10.63638629]
n=8, h=0.5 		 [-11.27621691]
n=8, h=2 		 [-13.1308717]
n=8, h=5 		 [-32.97485968]
n=5, h=0 		 [-6.39785998]
n=5, h=0.5 		 [-6.6102764]
n=5, h=2 		 [-8.47608351]
n=5, h=5 		 [-20.99943055]
n=3, h=0 		 [-3.40798777]
n=3, h=0.5 		 [-4.38379996]
n=3, h=2 		 [-5.04254922]
n=3, h=5 		 [-12.96603707]


In [4]:
num_sites = [5, 12]
couplings = [0.5, 5]

final_time = 0.25
num_timesteps = 5

In [5]:
estimator = StatevectorEstimator()

for (n, h) in product(num_sites, couplings) :
    dummy = QuantumCircuit(n)
    H = hamiltonian_generator.get_heisenberg_hamiltonian(n, h)
    sim_energy = (
        estimator.run([(dummy, H)]).result()[0].data.evs
    )
    print(f"n={n}, h={h}: \t\t {sim_energy}")
    np.save(f'data/evo_ref_{n}_{h}', sim_energy)

n=5, h=0.5: 		 6.5
n=5, h=5: 		 29.0
n=12, h=0.5: 		 17.0
n=12, h=5: 		 71.0
